## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Setup & Auth

In [ ]:
# @title Connect to Google Drive {"vertical-output":true,"single-column":true,"display-mode":"code"}

from google.colab import drive
import os

# --- SETUP & AUTH ---
# Use force_remount=True to attempt a fresh connection if it previously failed
try:
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully.")
except Exception as e:
    print(f"Manual action required: Please click the Drive icon in the left file pane to mount your drive. Error: {e}")

# --- CONFIGURATION (Master Variables) ---
global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

SOURCE_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step05"}
NEXT_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/bigquery_upload_step06" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/bigquery_upload_step06"}

NEW_DIR = os.path.join(SOURCE_DIR, "data_upload")
PROCESSED_DIR = os.path.join(SOURCE_DIR, "data_processed")
EXPORT_DIR = os.path.join(SOURCE_DIR, "data_export")
FAILED_DIR = os.path.join(SOURCE_DIR, "data_failed")
NEXT_DIR = os.path.join(NEXT_DIR, "data_upload")

# Create the necessary folders if they don't exist
if not os.path.exists(NEW_DIR):
    os.makedirs(NEW_DIR)
    print(f"Created directory: {NEW_DIR}")
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)
    print(f"Created directory: {PROCESSED_DIR}")
if not os.path.exists(EXPORT_DIR):
    os.makedirs(EXPORT_DIR)
    print(f"Created directory: {EXPORT_DIR}")
if not os.path.exists(FAILED_DIR):
    os.makedirs(FAILED_DIR)
    print(f"Created directory: {FAILED_DIR}")

print(f"Checking for new files in: {NEW_DIR}")

Mounted at /content/drive
Drive mounted successfully.
Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_upload


# Mapping Tables

In [ ]:
import io
import pandas as pd
segment_mapping_df = pd.read_csv(io.StringIO('''
segment_code,segment_name,segment_group_code,segment_group_name,segment_sort
RE,Transient Retail,TRE,Transient Retail,11
CN,Transient Consortia,TNG,Transient Negotiated,12
NG,Transient Negotiated,TNG,Transient Negotiated,13
QD,Transient Qualified,TQD,Transient Qualified,15
GV,Transient Government,TQD,Transient Qualified,16
PR,Transient Promotion,TQD,Transient Qualified,17
UQ,Transient Unqualified,TUQ,Transient Discount,17
PK,Transient Package,TQD,Transient Qualified,18
WH,Transient Wholesale,TWH,Transient Wholesale,19
OP,Transient Opaque,TUQ,Transient Discount,20
GR,Group,GGG,Group,30
GA,Group Association,GGA,Group Association,31
GY,Group Citywide,GGA,Group Association,32
CY,Group Citywide,GGA,Group Association,32
GC,Group Corporate,GGC,Group Corporate,33
GI,Group Incentive,GGS,Group SMERF,34
GT,Group Tour,GGT,Group Tour,36
GG,Group Government,GGG,Group Government,37
GS,Group SMERF,GGS,Group SMERF,46
GO,Group Social,GGS,Group SMERF,47
GE,Group Entertainment,GGS,Group SMERF,48
GW,Group Wedding,GGS,Group SMERF,49
CT,Contract,CCT,Contract,50
CO,Complimentary,CCO,Complimentary,60
HS,House,CCO,Complimentary,70
NS,No Show,RNS,Other Room Revenue,80
'''))
display(segment_mapping_df.head())

,segment_code,segment_name,segment_group_code,segment_group_name,segment_sort
0,RE,Transient Retail,TRE,Transient Retail,11
1,CN,Transient Consortia,TNG,Transient Negotiated,12
2,NG,Transient Negotiated,TNG,Transient Negotiated,13
3,QD,Transient Qualified,TQD,Transient Qualified,15
4,GV,Transient Government,TQD,Transient Qualified,16


In [ ]:
import io
import pandas as pd
pms_rate_mapping_df = pd.read_csv(io.StringIO('''
"ADV,Plan Ahead and Save"
"BAR,Best Flexible Rate"
"BUDDY,Buddy Buddy Opening Promo"
"CCRP1,CCRP1"
"COMP,Complimentary"
"CREATENOW,Influencer Discount Code"
"DISC20,Limited Time Offer"
"DISC402NT,Summer Offer - July / August"
"DISC252NT, Summer Offer"
"DISBDAY,It's Our Birthday!"
"DISCYBER,Cyber Sale"
"DISCYBERM,Cyber Sale - Book Direct + Save"
"DISDIR,Pre-Pay and Save"
"DISDIRMEM,Book Direct: Pre-pay and Save"
"DISFALL,Fall into NYC"
"DISFALLWE,Book Direct + Save with Perks! - Fall Into NYC"
"DISINTL,International Travelers"
"DISLOS,Stay Longer and Save"
"DISLOSMEM,Book Direct: Stay Longer and Save"
"DISLTO,Limited Time Offer"
"DISMBDAY,Members: It's Our Birthday!"
"DISWEB,Website Subscriber"
"DISWED,Wedding Rate"
"DISWINMEM,Winter in NYC - Book Direct + Save (with Perks!)"
"DISWINTER,Winter in NYC"
"DSADVMEM,Book Direct: Plan Ahead and Save"
"DUKEU260327-164218,Duke University- Oceans group"
"EC,Expedia Package"
"EC,Expedia Promotion LTO"
"EC,Expedia Standard"
"FITTFN,Travel Funders Network - Standard"
"FITTFNPRO,Travel Funders Network - Promotion"
"FITTNR,Travel Funders Network - Non-Refundable"
"FITTSR,Travel Funders Network - Static"
"HC,Expedia Promotion LTO"
"HC,Expedia Standard"
"HTLOS,Hotel Trader LOS"
"HTMEM,Hotel Trader - Member Discount"
"HTMEMP,Hotel Trader - Member Promotion"
"HTMEN,Hotel Trader - Member Discount NRF"
"HTPRO,Hotel Trader Advance Purchase"
"HTRET,Hotel Trader Standard Rate"
"HTRETN,Hotel Trader - NRF"
"MEMBERONLY,Book Direct + Save"
"MSUMUBER,Book Direct: SUMMER: Stay, Ride & Explore"
"NEGDEC1,Decarie Corporate Rate"
"NEGISIC,ISIC"
"NEGWMOD,Wilhelmina Model Perks"
"NGCREW,Airline Staff Rates"
"NGMADE,Made Worldwide Perks"
"NEGMADE,Made Worldwide Perks"
"NGRAIR,Republic Air"
"NOMAD260926-143810,Nomad Cruise"
"ODDIN251207-142640,Oddins"
"OTAGAP,Agoda Advance Purchase"
"OTAGLO,Agoda Promotion LOS"
"OTAGLT,Agoda Promotion LTO"
"OTAGNR,Agoda Non-Refundable"
"OTAGO,Agoda Retail"
"OTAGPKG,Agoda Package Rate"
"OTBKG,Booking.com Standard"
"OTBKGAP,Booking Advance Purchase"
"OTBKGIN,Booking International Rate"
"OTBKGLO,Booking Promotion LOS"
"OTBKGLT,Booking.com Promotion LTO"
"OTBKGNR,Booking Non-Refundable"
"OTEXAPE,Expedia Advance Purchase EC"
"OTEXINT,Expedia International"
"OTEXLOE,Expedia Promotion LOS EC"
"OTEXLOH,Expedia Promotion LOS HC"
"OTEXNRF,Expedia Non-Refundable"
"OTHCAP,Hopper CUG Advance Purchase"
"OTHCRE,Hopper CUG Fully Refundable"
"OTHOAP,Hopper Advance Purchase"
"OTHOLOS,Hopper Promotion LOS"
"OTHOP,Hopper Retail"
"OTHOS,Hostelworld Standard"
"OTHOSAP,Hostelworld Advance Purchase"
"OTHOSLO,Hostelworld Promotion LOS"
"OTHOSLT,Hostelworld Promotion LTO"
"OTRSNR,Rocket Stay Non-Refundable Rate"
"OTRSST,RocketStay Standard Rate"
"OTTRAP,Trip.com Advance Purchase - Trip Collect"
"OTTRAPHC,Trip.com Advance Purchase - Hotel Collect"
"OTTRIP,Trip.com Retail - Trip Collect"
"OTTRIPHC,Trip.com Retail - Hotel Collect"
"OTTRLO,Trip.com Promotion LOS - Trip Collect"
"OTTRLOHC,Trip.com Promotion LOS - Hotel Collect"
"OTTRLT,Trip.com Promotion LTO - Trip Collect"
"OTTRLTHC,Trip.com Promotion LTO - Hotel Collect"
"pms_rate_code,rate_name"
"SAVE20,Lock in 20%"
"SUMUBER,SUMMER: Stay, Ride & Explore"
"UNIVE260324-213409,University of Washington School of Arts"
"UPRIGHT,Upright Citizens Brigade Rate"
"WHATI,America Tours International Wholesale"
"WHDNA,DNATA"
"WHENG,Engine Supplemental"
"WHLOTS,Open Travel Service Standard Rate"
"WHNUT,Nuitee Standard Rate"
"WHOTSPRO,Open Travel Service Promotional Rate"
"WHW2M,W2M Travel Standard Rate"
"WHW2MNR,W2M Travel NR Rate"
"WHW2MPRO,W2M Travel NR Rate"
"WHWB,Webbeds Room Only"
"WHWBNRF,WebBeds Rate Room Only Non-Ref B2B&B2C"
"WHWBPRO,WebBeds Promo RO Type 1"
"with Perks!,Treat Yourself Now - Book Direct + Save"
"WLOTSNRF,Open Travel Service - Non-Refundable Rate"
'''))
display(pms_rate_mapping_df.head())

,"ADV,Plan Ahead and Save"
0,"BAR,Best Flexible Rate"
1,"BUDDY,Buddy Buddy Opening Promo"
2,"CCRP1,CCRP1"
3,"COMP,Complimentary"
4,"CREATENOW,Influencer Discount Code"


In [ ]:
import io
import pandas as pd
source_mapping_df = pd.read_csv(io.StringIO('''
source_code,source_name
HD,Hotel Direct
HO,Hopper
BK,Booking
EX,Expedia
TN,TFN Holidays
CT,Trip
HD,Direct
BK,Booking
EX,Expedia
HO,Hopper
CT,Trip
OE,Open Travel Service
AG,Agoda
WE,Webbeds
NU,Nuitee
RS,Rocket Stay
TN,TFN Holidays
W2,World2Meet
WB,Desktop
HL,Hostelworld
MB,Mobile
AA,Sabre
1A,Amadeus
AT,American Tours
TR,HotelTrader
DN,Dnata
ES,Engine Supplemental
TW,WorldSpan
'''))
display(source_mapping_df.head())

,source_code,source_name
0,HD,Hotel Direct
1,HO,Hopper
2,BK,Booking
3,EX,Expedia
4,TN,TFN Holidays


In [ ]:
import io
import pandas as pd
channel_mapping_df = pd.read_csv(io.StringIO('''
channel_code,channel_name
HD,Hotel Direct
DC,Direct Connect
BE,Booking Engine
GD,Global Distribution System
'''))
display(channel_mapping_df.head())

,channel_code,channel_name
0,HD,Hotel Direct
1,DC,Direct Connect
2,BE,Booking Engine
3,GD,Global Distribution System


In [ ]:
import io
import pandas as pd
subsource_mapping_df = pd.read_csv(io.StringIO('''
subsource_code,subsource_name
IH,In-House Reservations
EX,Expedia
HT,Hotel Tonight
WE,Webbeds
TN,TFN Holidays
WB,Desktop
MB,Mobile
AA,Sabre
1A,Amadeus
BO,Bonotel
AL,ALG Vacations
WH,FIT / Wholesale
TW,WorldSpan
HO,Hopper
HD,Hotel Direct
HO,Hopper
BK,Booking
EX,Expedia
TN,TFN Holidays
CT,Trip
HD,Direct
BK,Booking
EX,Expedia
HO,Hopper
CT,Trip
OE,Open Travel Service
AG,Agoda
WE,Webbeds
NU,Nuitee
RS,Rocket Stay
TN,TFN Holidays
W2,World2Meet
WB,Desktop
HL,Hostelworld
MB,Mobile
AA,Sabre
1A,Amadeus
AT,American Tours
TR,HotelTrader
DN,Dnata
ES,Engine Supplemental
TW,WorldSpan
'''))
display(subsource_mapping_df.head())

,subsource_code,subsource_name
0,IH,In-House Reservations
1,EX,Expedia
2,HT,Hotel Tonight
3,WE,Webbeds
4,TN,TFN Holidays


# 1. Load Your Raw Data Files

To begin, you'll need to load the data files you wish to amend. These files are typically located in the `NEW_DIR` directory (which is currently set to `'/content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step04/data_upload'`).

Please replace the placeholder code in the next cell with your actual file loading logic. You can load CSV, Excel, or other formats using pandas. Make sure to assign your loaded data to a variable, for example, `raw_data_df`.

---

### 2. Amend Data Using Mapping Tables

After loading your raw data, you can use the mapping tables (e.g., `segment_mapping_df`, `rate_mapping_df`) to add new columns or update existing ones based on matching keys. The `amend_data_file` function provided below will help you perform `left` merges.

For each mapping table, you will need to:
1.  **Identify the common column(s)** between your `raw_data_df` and the specific `_mapping_df` (e.g., 'segment_code' for `segment_mapping_df`).
2.  **Call the `amend_data_file` function** with your `raw_data_df`, the desired `_mapping_df`, and the common column name.

In [ ]:
import pandas as pd
import os
import glob
import shutil
import io

# --- MODULE: Data Amendment Logic ---
def amend_data_file(data_df: pd.DataFrame, mapping_df: pd.DataFrame, on_column: str) -> pd.DataFrame:
    """Performs a left merge to add mapping columns to the main dataset."""
    if on_column not in data_df.columns:
        print(f"... Skipping: '{on_column}' not in file.")
        return data_df

    cols_to_add = [col for col in mapping_df.columns if col != on_column and col not in data_df.columns]
    if not cols_to_add:
        return data_df

    mapping_subset = mapping_df[[on_column] + cols_to_add].drop_duplicates(subset=[on_column])
    amended_df = pd.merge(data_df, mapping_subset, on=on_column, how='left')
    print(f"... Added: {cols_to_add} via '{on_column}'")
    return amended_df

# --- MASTER PIPELINE: Execution Controller ---
def run_mapping_pipeline():
    print("### STARTING MASTER PIPELINE: Mapping & Amendment ###")

    # 1. Validation: Ensure required mapping globals exist in the environment
    required_mappings = {
        'segment_mapping_df': segment_mapping_df,
        'pms_rate_mapping_df': pms_rate_mapping_df,
        'source_mapping_df': source_mapping_df,
        'channel_mapping_df': channel_mapping_df,
        'subsource_mapping_df': subsource_mapping_df
    }

    # Ensure handoff directory exists
    if not os.path.exists(NEXT_DIR):
        os.makedirs(NEXT_DIR)
        print(f"Created handoff directory: {NEXT_DIR}")

    # 2. Identify Files
    files_to_process = glob.glob(os.path.join(NEW_DIR, "*.csv")) + glob.glob(os.path.join(NEW_DIR, "*.xlsx"))
    if not files_to_process:
        print(f"No new files found in: {NEW_DIR}")
        return

    # 3. Main Processing Loop
    for file_path in files_to_process:
        file_name = os.path.basename(file_path)
        print(f"\n--- Processing: {file_name} ---")
        try:
            df = pd.read_csv(file_path, low_memory=False) if file_path.endswith('.csv') else pd.read_excel(file_path)

            # Apply Transformations using localized references to the mapping DFs
            df = amend_data_file(df, segment_mapping_df, 'segment_code')
            df = amend_data_file(df, pms_rate_mapping_df, 'pms_rate_code')
            df = amend_data_file(df, source_mapping_df, 'source_code')
            df = amend_data_file(df, channel_mapping_df, 'channel_code')
            df = amend_data_file(df, subsource_mapping_df, 'subsource_code')

            # Save Locally to EXPORT_DIR
            export_name = f"amended_{file_name}"
            export_path = os.path.join(EXPORT_DIR, export_name)
            df.to_csv(export_path, index=False)
            print(f"Success: Saved to {EXPORT_DIR}")

            # Handoff: Copy to NEXT_DIR for process_step06
            handoff_path = os.path.join(NEXT_DIR, export_name)
            shutil.copy2(export_path, handoff_path)
            print(f"Handoff: Copied to {NEXT_DIR}")

            # Archive Original
            shutil.move(file_path, os.path.join(PROCESSED_DIR, file_name))
            display(df.head(3))

        except Exception as e:
            print(f"FAILED {file_name}: {e}")
            # Move to failed directory if error occurs
            shutil.move(file_path, os.path.join(FAILED_DIR, file_name))

    print("\n### PIPELINE COMPLETE ###")

run_mapping_pipeline()

### STARTING MASTER PIPELINE: Mapping & Amendment ###

--- Processing: 20260702_JFKNOW_standardized.csv ---
... Added: ['segment_name', 'segment_group_code', 'segment_group_name', 'segment_sort'] via 'segment_code'
... Added: ['rate_name'] via 'pms_rate_code'
... Added: ['source_name'] via 'source_code'
... Added: ['channel_name'] via 'channel_code'
... Added: ['subsource_name'] via 'subsource_code'
Success: Saved to /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_export
FAILED 20260702_JFKNOW_standardized.csv: [Errno 2] No such file or directory: '/content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step06/data_upload/amended_20260702_JFKNOW_standardized.csv'

### PIPELINE COMPLETE ###


In [ ]:
# --- RECOVERY UTILITY: Restore Files for Re-processing ---
def restore_processed_files():
    """Utility to move files from processed back to upload for re-runs."""
    processed_files = glob.glob(os.path.join(PROCESSED_DIR, "*"))

    if not processed_files:
        print(f"No files found in processed directory: {PROCESSED_DIR}")
        return

    print(f"Found {len(processed_files)} files in processed folder.")
    for f in processed_files:
        dest = os.path.join(NEW_DIR, os.path.basename(f))
        shutil.move(f, dest)
        print(f"Restored: {os.path.basename(f)} -> data_upload")

# To restore files, uncomment the line below and run this cell:
# restore_processed_files()